In [30]:
# Imports
import json
import concurrent.futures
import re
import os 
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from anthropic import Anthropic

In [31]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv(override=True)

client = Anthropic(
    base_url=os.getenv("ANTHROPIC_BASE_URL"),
    api_key=os.getenv("ANTHROPIC_AUTH_TOKEN")
)

model_name = os.getenv("ANTHROPIC_MODEL")

print("Base URL:", os.getenv("ANTHROPIC_BASE_URL"))
print("Model:", model_name)
print("Authentication configured:", bool(os.getenv("ANTHROPIC_AUTH_TOKEN")))

Base URL: http://localhost:11434
Model: qwen2.5:3b
Authentication configured: True


In [32]:
# helper functions 

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model_name,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [33]:
# Report Builder
def generate_prompt_evaluation_report(evaluation_results):
    total_tests = len(evaluation_results)
    scores = [result["score"] for result in evaluation_results]
    avg_score = mean(scores) if scores else 0
    max_possible_score = 10
    pass_rate = (
        100 * len([s for s in scores if s >= 7]) / total_tests if total_tests else 0
    )

    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prompt Evaluation Report</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                color: #333;
            }}
            .header {{
                background-color: #f0f0f0;
                padding: 20px;
                border-radius: 5px;
                margin-bottom: 20px;
            }}
            .summary-stats {{
                display: flex;
                justify-content: space-between;
                flex-wrap: wrap;
                gap: 10px;
            }}
            .stat-box {{
                background-color: #fff;
                border-radius: 5px;
                padding: 15px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                flex-basis: 30%;
                min-width: 200px;
            }}
            .stat-value {{
                font-size: 24px;
                font-weight: bold;
                margin-top: 5px;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin-top: 20px;
            }}
            th {{
                background-color: #4a4a4a;
                color: white;
                text-align: left;
                padding: 12px;
            }}
            td {{
                padding: 10px;
                border-bottom: 1px solid #ddd;
                vertical-align: top;
            }}
            tr:nth-child(even) {{
                background-color: #f9f9f9;
            }}
            .output-cell {{
                white-space: pre-wrap;
            }}
            .score {{
                font-weight: bold;
                padding: 5px 10px;
                border-radius: 3px;
                display: inline-block;
            }}
            .score-high {{
                background-color: #c8e6c9;
                color: #2e7d32;
            }}
            .score-medium {{
                background-color: #fff9c4;
                color: #f57f17;
            }}
            .score-low {{
                background-color: #ffcdd2;
                color: #c62828;
            }}
            .output {{
                overflow: auto;
                white-space: pre-wrap;
            }}

            .output pre {{
                background-color: #f5f5f5;
                border: 1px solid #ddd;
                border-radius: 4px;
                padding: 10px;
                margin: 0;
                font-family: 'Consolas', 'Monaco', 'Courier New', monospace;
                font-size: 14px;
                line-height: 1.4;
                color: #333;
                box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.1);
                overflow-x: auto;
                white-space: pre-wrap; 
                word-wrap: break-word; 
            }}

            td {{
                width: 20%;
            }}
            .score-col {{
                width: 80px;
            }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box">
                    <div>Total Test Cases</div>
                    <div class="stat-value">{total_tests}</div>
                </div>
                <div class="stat-box">
                    <div>Average Score</div>
                    <div class="stat-value">{avg_score:.1f} / {max_possible_score}</div>
                </div>
                <div class="stat-box">
                    <div>Pass Rate (≥7)</div>
                    <div class="stat-value">{pass_rate:.1f}%</div>
                </div>
            </div>
        </div>

        <table>
            <thead>
                <tr>
                    <th>Scenario</th>
                    <th>Prompt Inputs</th>
                    <th>Solution Criteria</th>
                    <th>Output</th>
                    <th>Score</th>
                    <th>Reasoning</th>
                </tr>
            </thead>
            <tbody>
    """

    for result in evaluation_results:
        prompt_inputs_html = "<br>".join(
            [
                f"<strong>{key}:</strong> {value}"
                for key, value in result["test_case"]["prompt_inputs"].items()
            ]
        )

        criteria_string = "<br>• ".join(result["test_case"]["solution_criteria"])

        score = result["score"]
        if score >= 8:
            score_class = "score-high"
        elif score <= 5:
            score_class = "score-low"
        else:
            score_class = "score-medium"

        html += f"""
            <tr>
                <td>{result["test_case"]["scenario"]}</td>
                <td class="prompt-inputs">{prompt_inputs_html}</td>
                <td class="criteria">• {criteria_string}</td>
                <td class="output"><pre>{result["output"]}</pre></td>
                <td class="score-col"><span class="score {score_class}">{score}</span></td>
                <td class="reasoning">{result["reasoning"]}</td>
            </tr>
        """

    html += """
            </tbody>
        </table>
    </body>
    </html>
    """

    return html

In [34]:
import concurrent.futures
import json
import re
from statistics import mean
from textwrap import dedent


class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks

    def render(self, template_string, variables):
        placeholders = re.findall(r"{([^{}]+)}", template_string)

        result = template_string

        for placeholder in placeholders:
            if placeholder in variables:
                result = result.replace(
                    "{" + placeholder + "}",
                    str(variables[placeholder]),
                )

        return result.replace("{{", "{").replace("}}", "}")

    def _parse_json(self, text):
        """Parse JSON returned by the model, including Markdown-wrapped JSON."""

        if not isinstance(text, str) or not text.strip():
            raise ValueError("The model returned an empty response.")

        text = text.strip()

        # Remove Markdown code fences if present.
        text = re.sub(
            r"^\s*```(?:json)?\s*",
            "",
            text,
            count=1,
            flags=re.IGNORECASE,
        )

        text = re.sub(
            r"\s*```\s*$",
            "",
            text,
            count=1,
        )

        text = text.strip()

        try:
            return json.loads(text)
        except json.JSONDecodeError as original_error:
            # Recover a JSON array or object if extra text surrounds it.
            decoder = json.JSONDecoder()

            for index, character in enumerate(text):
                if character not in "[{":
                    continue

                try:
                    parsed, _ = decoder.raw_decode(text[index:])
                    return parsed
                except json.JSONDecodeError:
                    continue

            raise ValueError(
                "Could not parse valid JSON from the model response. "
                f"Original error: {original_error}. "
                f"Response received: {text!r}"
            ) from original_error

    def generate_unique_ideas(
        self,
        task_description,
        prompt_inputs_spec,
        num_cases,
    ):
        """Generate unique ideas for test cases."""

        prompt = """
        Generate {num_cases} unique, diverse ideas for testing a prompt
        that accomplishes this task:

        <task_description>
        {task_description}
        </task_description>

        The prompt will receive the following inputs:

        <prompt_inputs>
        {prompt_inputs_spec}
        </prompt_inputs>

        Each idea should represent a distinct scenario or example.

        Requirements:
        - Each idea must be distinct and relevant.
        - Each idea must be specific enough to guide test-case generation.
        - Each case should be quick to solve.
        - Remember to generate exactly {num_cases} ideas.

        Output requirements:
        - Return only a valid JSON array.
        - Each array element must be a string.
        - Do not include Markdown code fences.
        - Do not include explanations outside the JSON array.

        Example:
        [
            "Testing a beginner athlete with a muscle-gain goal",
            "Testing an endurance athlete with dietary restrictions"
        ]
        """

        system_prompt = (
            "You are a test scenario designer specializing in creating "
            "diverse, unique testing scenarios. Return valid JSON only."
        )

        example_prompt_inputs = {}

        for key, value in prompt_inputs_spec.items():
            example_prompt_inputs[key] = value

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "task_description": task_description,
                "num_cases": num_cases,
                "prompt_inputs_spec": json.dumps(
                    example_prompt_inputs,
                    indent=2,
                ),
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)

        text = chat(
            messages,
            system=system_prompt,
            temperature=0.0,
        )

        ideas = self._parse_json(text)

        if not isinstance(ideas, list):
            raise ValueError(
                "Expected a JSON array when generating test-case ideas."
            )

        return ideas[:num_cases]

    def generate_test_case(
        self,
        task_description,
        idea,
        prompt_inputs_spec=None,
    ):
        """Generate a single test case from a specific idea."""

        if prompt_inputs_spec is None:
            prompt_inputs_spec = {}

        allowed_keys = list(prompt_inputs_spec.keys())

        example_prompt_inputs = "\n".join(
            f'"{key}": "EXAMPLE_VALUE"'
            for key in allowed_keys
        )

        prompt = """
        Generate a single detailed test case for a prompt evaluation.

        <task_description>
        {task_description}
        </task_description>

        <specific_idea>
        {idea}
        </specific_idea>

        <allowed_input_keys>
        {allowed_keys}
        </allowed_input_keys>

        The test case must follow this JSON structure:

        {{
            "prompt_inputs": {{
                {example_prompt_inputs}
            }},
            "solution_criteria": [
                "criterion 1",
                "criterion 2"
            ]
        }}

        Requirements:
        - Use exactly the input keys listed in allowed_input_keys.
        - Include every allowed input key.
        - Do not add any extra keys inside prompt_inputs.
        - Generate realistic and practically useful input values.
        - Include 1 to 4 concise, measurable solution criteria.
        - Criteria must address only the task and generated inputs.
        - Keep the test case focused and straightforward.
        - Return only a valid JSON object.
        - Do not use Markdown code fences.
        - Do not include comments or explanations outside the JSON.
        """

        system_prompt = (
            "You are a test-case creator specializing in evaluation "
            "scenarios. Return valid JSON only, with no Markdown fences."
        )

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "allowed_keys": json.dumps(allowed_keys),
                "task_description": task_description,
                "idea": idea,
                "example_prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)

        text = chat(
            messages,
            system=system_prompt,
            temperature=0.0,
        )

        test_case = self._parse_json(text)

        if not isinstance(test_case, dict):
            raise ValueError(
                "Expected a JSON object when generating a test case."
            )

        if "prompt_inputs" not in test_case:
            raise ValueError(
                "Generated test case is missing 'prompt_inputs'."
            )

        if "solution_criteria" not in test_case:
            raise ValueError(
                "Generated test case is missing 'solution_criteria'."
            )

        # Validate that the model returned exactly the requested input keys.
        generated_keys = set(test_case["prompt_inputs"].keys())
        expected_keys = set(allowed_keys)

        if generated_keys != expected_keys:
            raise ValueError(
                "Generated prompt_inputs keys do not match the requested "
                f"keys. Expected: {sorted(expected_keys)}; "
                f"received: {sorted(generated_keys)}"
            )

        test_case["task_description"] = task_description
        test_case["scenario"] = idea

        return test_case

    def generate_dataset(
        self,
        task_description,
        prompt_inputs_spec=None,
        num_cases=1,
        output_file="dataset.json",
    ):
        """Generate test cases and save the dataset to a JSON file."""

        if prompt_inputs_spec is None:
            prompt_inputs_spec = {}

        if num_cases < 1:
            raise ValueError("num_cases must be at least 1.")

        ideas = self.generate_unique_ideas(
            task_description,
            prompt_inputs_spec,
            num_cases,
        )

        if not ideas:
            raise ValueError("The model did not generate any test ideas.")

        dataset = []
        completed = 0
        total = len(ideas)

        # Use a thread pool to generate test cases.
        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:

            future_to_idea = {
                executor.submit(
                    self.generate_test_case,
                    task_description,
                    idea,
                    prompt_inputs_spec,
                ): idea
                for idea in ideas
            }

            for future in concurrent.futures.as_completed(future_to_idea):
                idea = future_to_idea[future]

                try:
                    result = future.result()
                    dataset.append(result)
                    completed += 1

                    print(
                        f"Generated {completed}/{total} test cases"
                    )

                except Exception as error:
                    print(
                        f"Error generating test case for {idea!r}: "
                        f"{error}"
                    )

        # Save only successfully generated cases.
        with open(output_file, "w", encoding="utf-8") as file:
            json.dump(dataset, file, indent=2, ensure_ascii=False)

        print(
            f"Saved {len(dataset)} test cases to {output_file}"
        )

        return dataset

    def grade_output(self, test_case, output, extra_criteria=None):
        """Grade a model-generated solution against evaluation criteria."""

        prompt_inputs = json.dumps(
            test_case["prompt_inputs"],
            indent=2,
            ensure_ascii=False,
        )

        extra_criteria_section = ""

        if extra_criteria:
            extra_criteria_template = """
            Mandatory Requirements:
            Any violation means automatic failure with a score of 3 or lower.

            <extra_important_criteria>
            {extra_criteria}
            </extra_important_criteria>
            """

            extra_criteria_section = self.render(
                dedent(extra_criteria_template),
                {"extra_criteria": extra_criteria},
            )

        eval_template = """
        Evaluate the following AI-generated solution rigorously.

        Original task description:
        <task_description>
        {task_description}
        </task_description>

        Original task inputs:
        <task_inputs>
        {prompt_inputs}
        </task_inputs>

        Solution to evaluate:
        <solution>
        {output}
        </solution>

        Evaluation criteria:
        <criteria>
        {solution_criteria}
        </criteria>

        {extra_criteria_section}

        Scoring guidelines:
        - Score 1-3: One or more mandatory requirements are violated.
        - Score 4-6: Mandatory requirements are met, but significant
          deficiencies remain.
        - Score 7-8: Requirements are met with minor issues.
        - Score 9-10: All listed requirements are satisfied.

        Grade only against the supplied criteria.
        Do not introduce additional requirements.
        If all requirements are satisfied, give a score of 10.
        Any violation of an extra mandatory requirement must result
        in a score of 3 or lower.

        Return only a valid JSON object with these fields:
        {{
            "strengths": ["strength 1"],
            "weaknesses": ["weakness 1"],
            "reasoning": "Concise explanation",
            "score": 10
        }}

        The score must be a number between 1 and 10.
        Do not include Markdown fences or explanations outside the JSON.
        """

        eval_prompt = self.render(
            dedent(eval_template),
            {
                "task_description": test_case["task_description"],
                "prompt_inputs": prompt_inputs,
                "output": output,
                "solution_criteria": "\n".join(
                    test_case["solution_criteria"]
                ),
                "extra_criteria_section": extra_criteria_section,
            },
        )

        messages = []
        add_user_message(messages, eval_prompt)

        eval_text = chat(
            messages,
            system=(
                "You are a rigorous prompt evaluator. "
                "Return only a valid JSON object."
            ),
            temperature=0.0,
        )

        model_grade = self._parse_json(eval_text)

        required_fields = {
            "strengths",
            "weaknesses",
            "reasoning",
            "score",
        }

        if not required_fields.issubset(model_grade.keys()):
            raise ValueError(
                "The grading response is missing required fields: "
                f"{required_fields - set(model_grade.keys())}"
            )

        score = model_grade["score"]

        if isinstance(score, bool) or not isinstance(score, (int, float)):
            raise ValueError("The grading score must be numeric.")

        if not 1 <= score <= 10:
            raise ValueError("The grading score must be between 1 and 10.")

        return model_grade

    def run_test_case(
        self,
        test_case,
        run_prompt_function,
        extra_criteria=None,
    ):
        """Run one test case and grade the result."""

        output = run_prompt_function(test_case["prompt_inputs"])

        model_grade = self.grade_output(
            test_case,
            output,
            extra_criteria,
        )

        return {
            "output": output,
            "test_case": test_case,
            "score": model_grade["score"],
            "reasoning": model_grade["reasoning"],
            "grade": model_grade,
        }

    def run_evaluation(
        self,
        run_prompt_function,
        dataset_file,
        extra_criteria=None,
        json_output_file="output.json",
        html_output_file="output.html",
    ):
        """Evaluate all test cases and save JSON and HTML reports."""

        with open(dataset_file, "r", encoding="utf-8") as file:
            dataset = json.load(file)

        if not dataset:
            raise ValueError(
                f"No test cases found in {dataset_file}."
            )

        results = []
        completed = 0
        total = len(dataset)

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:

            future_to_test_case = {
                executor.submit(
                    self.run_test_case,
                    test_case,
                    run_prompt_function,
                    extra_criteria,
                ): test_case
                for test_case in dataset
            }

            for future in concurrent.futures.as_completed(
                future_to_test_case
            ):
                try:
                    result = future.result()
                    results.append(result)
                    completed += 1

                    print(f"Graded {completed}/{total} test cases")

                except Exception as error:
                    test_case = future_to_test_case[future]

                    print(
                        "Error evaluating test case "
                        f"{test_case.get('scenario', 'unknown')!r}: "
                        f"{error}"
                    )

        if not results:
            raise RuntimeError(
                "No test cases were evaluated successfully."
            )

        average_score = mean(
            result["score"] for result in results
        )

        print(f"Average score: {average_score:.2f}/10")

        with open(
            json_output_file,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                results,
                file,
                indent=2,
                ensure_ascii=False,
            )

        html = generate_prompt_evaluation_report(results)

        with open(
            html_output_file,
            "w",
            encoding="utf-8",
        ) as file:
            file.write(html)

        print(f"JSON report saved to {json_output_file}")
        print(f"HTML report saved to {html_output_file}")

        return results

In [35]:
# Create an instance of PromptEvaluator
# Increase `max_concurrent_tasks` for greater concurrency, but beware of rate limit errors!
evaluator = PromptEvaluator(max_concurrent_tasks=1)

In [36]:
dataset = evaluator.generate_dataset(
    # Describe the purpose or goal of the prompt you're trying to test
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    # Describe the different inputs that your prompt requires
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete",
    },
    # Where to write the generated dataset
    output_file="dataset.json",
    # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
    num_cases=1,
)

Generated 1/1 test cases
Saved 1 test cases to dataset.json


In [37]:
# Define and run the prompt you want to evaluate, returning the raw model output
# This function is executed once for each test case
def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a one-day meal plan for an athlete that meets their dietary restrictions.

    <athlete_information> 
    - Height: {prompt_inputs["height"]} 
    - Weight: {prompt_inputs["weight"]} 
    - Goal: {prompt_inputs["goal"]} 
    - Dietary restrictions: {prompt_inputs["restrictions"]} 
    </athlete_information>

    Guidelines:
    1. Include accurate daily calorie amount
    2. Show protein, fat, and carb amounts
    3. Specify when to eat each meal
    4. Use only foods that fit restrictions
    5. List all portion sizes in grams
    6. Keep budget-friendly if mentioned

    Here is an example with a sample input and an ideal output:
    <sample_input>
    height: 170
    weight: 70
    goal: Maintain fitness and improve cholesterol levels
    restrictions: High cholesterol
    </sample_input>
    <ideal_output>
    Here is a one-day meal plan for an athlete aiming to maintain fitness and improve cholesterol levels:

    *   **Calorie Target:** Approximately 2500 calories
    *   **Macronutrient Breakdown:** Protein (140g), Fat (70g), Carbs (340g)

    **Meal Plan:**

    *   **Breakfast (7:00 AM):** Oatmeal (80g dry weight) with berries (100g) and walnuts (15g). Skim milk (240g).
        *   Protein: 15g, Fat: 15g, Carbs: 60g
    *   **Mid-Morning Snack (10:00 AM):** Apple (150g) with almond butter (30g).
        *   Protein: 7g, Fat: 18g, Carbs: 25g
    *   **Lunch (1:00 PM):** Grilled chicken breast (120g) salad with mixed greens (150g), cucumber (50g), tomato (50g), and a light vinaigrette dressing (30g). Whole wheat bread (60g).
        *   Protein: 40g, Fat: 15g, Carbs: 70g
    *   **Afternoon Snack (4:00 PM):** Greek yogurt (170g, non-fat) with a banana (120g).
        *   Protein: 20g, Fat: 0g, Carbs: 40g
    *   **Dinner (7:00 PM):** Baked salmon (140g) with steamed broccoli (200g) and quinoa (75g dry weight).
        *   Protein: 40g, Fat: 20g, Carbs: 80g
    *   **Evening Snack (9:00 PM):** Small handful of almonds (20g).
        *   Protein: 8g, Fat: 12g, Carbs: 15g

    This meal plan prioritizes lean protein sources, whole grains, fruits, and vegetables, while limiting saturated and trans fats to support healthy cholesterol levels.
    </ideal_output>
    This example meal plan is well-structured, provides detailed information on food choices and quantities, and aligns with the athlete's goals and restrictions.
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [38]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)

Graded 1/1 test cases
Average score: 9.00/10
JSON report saved to output.json
HTML report saved to output.html
